In [21]:
import os
import glob
import re
import tempfile
import xarray as xr
from cdo import Cdo

In [22]:
cdo = Cdo(cdo='/usr/local/apps/cdo/2.4.0/bin/cdo')

# Helper functions

In [ ]:
def annual_mean_moc(input_folder, output_folder,
                    filename_filter, variables):
    """
    Compute annual means for zonal-mean diagnostics (msftyz), and save a single output file.

    Parameters
    ----------
    input_folder : str
        Folder containing the original NEMO monthly files.

    output_folder : str
        Folder where the final file will be written.

    filename_filter : str
        Pattern identifying the desired files
        (e.g. "oce_1m_diaptr3d").

    variables : list[str]
        Variables to retain (e.g. ["msftyz"]).
    """

    os.makedirs(output_folder, exist_ok=True)

    files = sorted(
        glob.glob(os.path.join(input_folder, f"*{filename_filter}*.nc"))
    )

    if len(files) == 0:
        print("No matching files found.")
        return

    annual_list = []

    years = []

    for file in files:

        print(f"Processing {os.path.basename(file)}")

        with xr.open_dataset(file) as ds:

            annual = (
                ds[variables]
                .mean(dim="time_counter", keep_attrs=True)
                .expand_dims(
                    time_counter=[ds.time_counter.values[0]]
                )
            )

            # Load the small annual dataset into memory so
            # the file can be closed immediately.
            annual.load()

            annual_list.append(annual)

        # Extract year from filename
        match = re.search(r"(\d{4})-(\d{4})", os.path.basename(file))
        if match:
            years.append(int(match.group(1)))

    # Concatenate all annual means
    final = xr.concat(annual_list, dim="time_counter")

    first_year = min(years)
    last_year = max(years)

    var_string = "_".join(variables)

    output_file = os.path.join(
        output_folder,
        f"moc_annual_mean_{first_year}-{last_year}.nc"
    )

    final.to_netcdf(output_file)

    print(f"\n Saved {output_file}")

In [23]:
def extract_year(filename):
    """
    Extract first year from filenames like:
    PV19_atm_cmip6_1m_1990-1990.nc
    """

    match = re.search(r"(\d{4})-\d{4}", filename)

    if match:
        return int(match.group(1))

    raise ValueError(
        f"Could not extract year from {filename}"
    )

In [ ]:
def process_dataset(cdo,input_folder,output_folder,filename_filter,variables,remap_method,output_prefix,exclude_filter=None,grid="r180x90"):
    """
    Select variables, calculate annual means, remap,
    concatenate all years and write one NetCDF file.

    Parameters
    ----------
    input_folder : str
        Folder containing monthly files.

    output_folder : str
        Where the final file is written.

    filename_filter : str
        String identifying input files.

    variables : str
        Variables separated by commas.

    remap_method : str
        "remapcon" or "remapbil".

    output_prefix : str
        Final filename prefix.
        Example: "atm" or "oce"

    exclude_filter : str
        Files containing this string are ignored.

    grid : str
        CDO target grid.
        r180x90 = 2° x 2°
    """

    os.makedirs(output_folder, exist_ok=True)

    files = sorted(
        glob.glob(
            os.path.join(
                input_folder,
                f"*{filename_filter}*.nc"
            )
        )
    )

    if exclude_filter:

        files = [
            f for f in files
            if exclude_filter not in os.path.basename(f)
        ]

    if len(files) == 0:
        print("No files found.")
        return

    print(
        f"Found {len(files)} files for {output_prefix}"
    )

    annual_list = []
    years = []

    for infile in files:

        filename = os.path.basename(infile)

        print(
            f"Processing {filename}"
        )


        year = extract_year(filename)
        years.append(year)


        # temporary files
        tmp_file = tempfile.NamedTemporaryFile(
            suffix=".nc",
            delete=False
        ).name

        cdo_command = (
            f"-{remap_method},{grid} "
            f"-yearmean "
            f"-selname,{variables} "
            f"{infile}"
        )


        cdo.copy(
            input=cdo_command,
            output=tmp_file
        )

        with xr.open_dataset(tmp_file) as ds:

            annual = ds.load()


        annual_list.append(annual)


        # remove temporary file
        os.remove(tmp_file)


    final = xr.concat(
        annual_list,
        dim="time_counter"
    )

    first_year = min(years)
    last_year = max(years)

    output_file = os.path.join(
        output_folder,
        f"{output_prefix}_annual_mean_{first_year}-{last_year}.nc"
    )

    if os.path.exists(output_file):

        print(
            f"Already exists: {output_file}"
        )

        return

    final.to_netcdf(
        output_file
    )

    print(
        f"\n Saved: {output_file}"
    )

In [ ]:
def annual_mean(input_folder, output_folder, cdo,
                filename_filter=None, exclude_filter=None):
    """
    Compute annual means from monthly NetCDF files using CDO yearmean.
    """

    os.makedirs(output_folder, exist_ok=True)

    for filename in os.listdir(input_folder):

        if not filename.endswith(".nc"):
            continue

        if filename_filter and filename_filter not in filename:
            continue

        if exclude_filter and exclude_filter in filename:
            continue

        input_file = os.path.join(input_folder, filename)
        output_file = os.path.join(output_folder, f"yearmean_{filename}")

        if not os.path.exists(output_file):

            cdo.yearmean(
                input=input_file,
                output=output_file
            )

            print(f" Annual mean: {filename}")

        else:
            print(f" Skipped: {filename}")

In [16]:
def remap_files(input_folder, output_folder, file_filter, method="remapbil"):
    """
    Remap netCDF files to a target grid.

    Parameters
    ----------
    input_folder : str
        Folder containing input NetCDF files.
    output_folder : str
        Folder for remapped files.
    file_filter : str
        String that must appear in the filename.
    method : str
        CDO remapping operator: "remapcon", "remapbil", "remapnn", etc.
    """

    os.makedirs(output_folder, exist_ok=True)

    for filename in os.listdir(input_folder):
        if filename.endswith(".nc") and file_filter in filename:

            input_file = os.path.join(input_folder, filename)
            output_file = os.path.join(output_folder, f"remapped_{filename}")

            if not os.path.exists(output_file):

                if method == "remapcon":
                    cdo.remapcon(
                        "r180x90",
                        input=input_file,
                        output=output_file
                    )

                elif method == "remapbil":
                    cdo.remapbil(
                        "r180x90",
                        input=input_file,
                        output=output_file
                    )

                elif method == "remapnn":
                    cdo.remapnn(
                        "r180x90",
                        input=input_file,
                        output=output_file
                    )

                else:
                    raise ValueError(f"Unknown remapping method: {method}")

                print(f"✅ Remapped: {filename}")

            else:
                print(f"⏩ Skipped (already exists): {filename}")

In [17]:
def extract_variables(cdo, input_folder, output_folder, variables,
                      filename_filter=None, exclude_filter=None,
                      year_range=None):
    """
    Extract selected variables from remapped files.
    Optional year filtering supported.
    """
    os.makedirs(output_folder, exist_ok=True)

    for filename in os.listdir(input_folder):
        if not filename.endswith(".nc"):
            continue

        if filename_filter and filename_filter not in filename:
            continue

        if exclude_filter and exclude_filter in filename:
            continue

        # Optional year filter
        if year_range:
            try:
                year_str = filename.split("_")[-1].replace(".nc", "")
                year = int(year_str.split("-")[0])
                if not (year_range[0] <= year <= year_range[1]):
                    continue
            except:
                continue

        infile = os.path.join(input_folder, filename)
        outfile = os.path.join(output_folder, f"small_{filename}")

        if not os.path.exists(outfile):
            cdo.selname(variables, input=infile, output=outfile)
            print(f"✅ Extracted vars from {filename}")
        else:
            print(f"⏩ Skipped (already exists): {filename}")

# Experiments configuration

In [12]:
experiments = {
    #"exp3": "/lus/h2resw01/scratch/itcv/ece4/exp3/output/",
    #"XPPI": "/lus/h2resw01/scratch/ccpd/ece4/XPPI/output/",
    #"XEPI": "/lus/h2resw01/scratch/ccpd/ece4/XEPI/output/",
    "XE3C": "/lus/h2resw01/scratch/ccpd/ece4/XE3C/output/",
    #"XE6C": "/lus/h2resw01/scratch/ccpd/ece4/XE6C/output/",
}

In [25]:
experiments = {"pex3": "/lus/h2resw01/scratch/ecme3497/ece4/pex3/output/"}

In [26]:
base_output = "/lus/h2resw01/hpcperm/ecme3497/data-analysis/epochal/"

atm_vars = "tas,pr,rsut,rlut,rsdt"
oce_vars = "tos,sos"
moc_vars = ["msftyz"]

In [20]:
for exp_name, input_base in experiments.items():

    annual_mean_moc(
        input_folder=os.path.join(input_base, "nemo"),
        output_folder=os.path.join(base_output, exp_name, "moc"),
        filename_filter="oce_1m_diaptr3d",
        variables=moc_vars
    )


Processing pex3_oce_1m_diaptr3d_1700-1700.nc
Processing pex3_oce_1m_diaptr3d_1701-1701.nc
Processing pex3_oce_1m_diaptr3d_1702-1702.nc
Processing pex3_oce_1m_diaptr3d_1703-1703.nc
Processing pex3_oce_1m_diaptr3d_1704-1704.nc
Processing pex3_oce_1m_diaptr3d_1705-1705.nc
Processing pex3_oce_1m_diaptr3d_1706-1706.nc
Processing pex3_oce_1m_diaptr3d_1707-1707.nc
Processing pex3_oce_1m_diaptr3d_1708-1708.nc
Processing pex3_oce_1m_diaptr3d_1709-1709.nc
Processing pex3_oce_1m_diaptr3d_1710-1710.nc
Processing pex3_oce_1m_diaptr3d_1711-1711.nc
Processing pex3_oce_1m_diaptr3d_1712-1712.nc
Processing pex3_oce_1m_diaptr3d_1713-1713.nc
Processing pex3_oce_1m_diaptr3d_1714-1714.nc
Processing pex3_oce_1m_diaptr3d_1715-1715.nc
Processing pex3_oce_1m_diaptr3d_1716-1716.nc
Processing pex3_oce_1m_diaptr3d_1717-1717.nc
Processing pex3_oce_1m_diaptr3d_1718-1718.nc
Processing pex3_oce_1m_diaptr3d_1719-1719.nc
Processing pex3_oce_1m_diaptr3d_1720-1720.nc
Processing pex3_oce_1m_diaptr3d_1721-1721.nc
Processing

# Main processing loop

In [27]:
for exp_name, input_base in experiments.items():

    print(f"\n================ {exp_name} ================\n")

    # Atmosphere
    process_dataset(
        cdo=cdo,
        input_folder=os.path.join(
            input_base,
            "oifs"
        ),
        output_folder=os.path.join(
            base_output,
            exp_name,
            "processed"
        ),
        filename_filter="atm_cmip6_1m",
        variables=atm_vars,
        remap_method="remapcon",
        output_prefix="atm",
        exclude_filter="_pl_"
    )

    # Ocean
    process_dataset(
        cdo=cdo,
        input_folder=os.path.join(
            input_base,
            "nemo"
        ),
        output_folder=os.path.join(
            base_output,
            exp_name,
            "processed"
        ),
        filename_filter="oce_1m_T",
        variables=oce_vars,
        remap_method="remapbil",
        output_prefix="oce"
    )


================ pex3 ================

Found 107 files for atm
Processing pex3_atm_cmip6_1m_1700-1700.nc
Processing pex3_atm_cmip6_1m_1701-1701.nc
Processing pex3_atm_cmip6_1m_1702-1702.nc
Processing pex3_atm_cmip6_1m_1703-1703.nc
Processing pex3_atm_cmip6_1m_1704-1704.nc
Processing pex3_atm_cmip6_1m_1705-1705.nc
Processing pex3_atm_cmip6_1m_1706-1706.nc
Processing pex3_atm_cmip6_1m_1707-1707.nc
Processing pex3_atm_cmip6_1m_1708-1708.nc
Processing pex3_atm_cmip6_1m_1709-1709.nc
Processing pex3_atm_cmip6_1m_1710-1710.nc
Processing pex3_atm_cmip6_1m_1711-1711.nc
Processing pex3_atm_cmip6_1m_1712-1712.nc
Processing pex3_atm_cmip6_1m_1713-1713.nc
Processing pex3_atm_cmip6_1m_1714-1714.nc
Processing pex3_atm_cmip6_1m_1715-1715.nc
Processing pex3_atm_cmip6_1m_1716-1716.nc
Processing pex3_atm_cmip6_1m_1717-1717.nc
Processing pex3_atm_cmip6_1m_1718-1718.nc
Processing pex3_atm_cmip6_1m_1719-1719.nc
Processing pex3_atm_cmip6_1m_1720-1720.nc
Processing pex3_atm_cmip6_1m_1721-1721.nc
Processing 

In [32]:
for exp_name, input_base in experiments.items():

    print(f"\n================ {exp_name} ================\n")

    # -------------------------
    # Extract ATM variables
    # -------------------------
    extract_variables(
        input_folder=os.path.join(input_base, "oifs"),
        output_folder=os.path.join(base_output, exp_name, "variables/atm"),
        variables=atm_vars,
        filename_filter="atm_cmip6_1m",
        exclude_filter="_pl_",
        cdo = cdo
    )

    # -------------------------
    # Extract OCE variables
    # -------------------------
    extract_variables(
        input_folder=os.path.join(input_base, "nemo"),
        output_folder=os.path.join(base_output, exp_name, "variables/oce"),
        variables=oce_vars,
        filename_filter="oce_1m_T",
        cdo=cdo
    )

    # ------------------------
    # Compute annual means
    # ------------------------
    annual_mean(
        input_folder=os.path.join(base_output, exp_name, "variables/atm"),
        output_folder=os.path.join(base_output, exp_name, "annual_mean/atm"),
        cdo=cdo
    )

    annual_mean(
        input_folder=os.path.join(base_output, exp_name, "variables/oce"),
        output_folder=os.path.join(base_output, exp_name, "annual_mean/oce"),
        cdo=cdo
    )

    # -------------------------
    # Remap atmosphere (OIFS)
    # -------------------------
    remap_files(
        input_folder=os.path.join(base_output, exp_name, "annual_mean/atm"),
        output_folder=os.path.join(base_output, exp_name, "remaped/atm"),
        file_filter="_1m_",
        method="remapcon"
    )

    # -------------------------
    # Remap ocean (NEMO)
    # -------------------------
    remap_files(
        input_folder=os.path.join(base_output, exp_name, "annual_mean/oce"),
        output_folder=os.path.join(base_output, exp_name, "remaped/oce"),
        file_filter="oce_1m_T",
        method="remapbil"
    )




================ PV19 ================

⏩ Skipped (already exists): PV19_atm_cmip6_1m_1995-1995.nc
⏩ Skipped (already exists): PV19_atm_cmip6_1m_1993-1993.nc
⏩ Skipped (already exists): PV19_atm_cmip6_1m_1992-1992.nc
⏩ Skipped (already exists): PV19_atm_cmip6_1m_1991-1991.nc
⏩ Skipped (already exists): PV19_atm_cmip6_1m_1990-1990.nc
⏩ Skipped (already exists): PV19_atm_cmip6_1m_1994-1994.nc
⏩ Skipped (already exists): PV19_oce_1m_T_1995-1995.nc
⏩ Skipped (already exists): PV19_oce_1m_T_1991-1991.nc
⏩ Skipped (already exists): PV19_oce_1m_T_1994-1994.nc
⏩ Skipped (already exists): PV19_oce_1m_T_1990-1990.nc
⏩ Skipped (already exists): PV19_oce_1m_T_1993-1993.nc
⏩ Skipped (already exists): PV19_oce_1m_T_1992-1992.nc
⏩ Skipped: small_PV19_atm_cmip6_1m_1993-1993.nc
⏩ Skipped: small_PV19_atm_cmip6_1m_1994-1994.nc
⏩ Skipped: small_PV19_atm_cmip6_1m_1992-1992.nc
⏩ Skipped: small_PV19_atm_cmip6_1m_1990-1990.nc
⏩ Skipped: small_PV19_atm_cmip6_1m_1995-1995.nc
⏩ Skipped: small_PV19_atm_cmip6_1m_